# 001 · Lists

> **Syntax › Collections** · Python 3.14

A `list` is Python's general-purpose, ordered, **mutable** sequence. It is the type you will reach for most often — the closest equivalent of C#'s `List<T>`.

**In this notebook you will learn:**

- how to create, read, and modify lists
- the most important list methods and their `List<T>` counterparts
- how lists differ from C# collections (mixed types, negative indexes, no fixed `T`)
- common gotchas: aliasing, shallow copies, and mutable default values

## C# mental model

| C# | Python |
|---|---|
| `List<int>` | `list[int]` (type hint only — not enforced at runtime) |
| `new List<int> { 1, 2, 3 }` | `[1, 2, 3]` |
| `list.Count` | `len(lst)` |
| `int[]` (fixed size array) | no direct equivalent — use `list` (or `array`/`tuple`) |

The key mental shift: a Python list holds **references to objects of any type**. There is no generic type parameter enforced by the runtime.

## Creating lists

```csharp
// C#
var numbers = new List<int> { 1, 2, 3 };
var empty = new List<string>();
var fromRange = Enumerable.Range(0, 5).ToList();
```

In Python you use square-bracket *literals*, or the `list()` constructor to convert any iterable:

In [ ]:
numbers = [1, 2, 3]
empty = []
from_range = list(range(5))        # range(5) is lazy; list() materialises it
from_string = list("abc")          # any iterable works

print(numbers, empty, from_range, from_string)

A list can hold mixed types. This is legal, but usually a code smell — if the items have different meanings, a `tuple` or a `dataclass` is clearer.

In [ ]:
mixed = [42, "hello", 3.14, None, [1, 2]]
print(mixed)
print(len(mixed))   # C#: mixed.Count

## Indexing — including negative indexes

Indexes start at 0 like in C#. Python also supports **negative indexes**, which count from the end. `lst[-1]` is the last element — similar to C#'s `list[^1]` index-from-end operator.

```csharp
// C#
var first = items[0];
var last  = items[^1];   // C# 8+
```

In [ ]:
items = ["a", "b", "c", "d"]

print(items[0])    # first
print(items[-1])   # last   (C#: items[^1])
print(items[-2])   # second to last

Going out of range raises an `IndexError` (C#: `ArgumentOutOfRangeException`):

In [ ]:
try:
    items[10]
except IndexError as e:
    print(f"{type(e).__name__}: {e}")

## Adding and removing items

| C# `List<T>` | Python `list` |
|---|---|
| `Add(x)` | `append(x)` |
| `AddRange(xs)` | `extend(xs)` or `+=` |
| `Insert(i, x)` | `insert(i, x)` |
| `Remove(x)` | `remove(x)` (raises `ValueError` if missing) |
| `RemoveAt(i)` | `del lst[i]` or `pop(i)` |
| `Clear()` | `clear()` |

In [ ]:
fruits = ["apple", "banana"]

fruits.append("cherry")              # add one item
fruits.extend(["date", "elderberry"]) # add many
fruits.insert(0, "avocado")          # insert at position
print(fruits)

fruits.remove("banana")              # remove first matching value
last = fruits.pop()                  # remove and return the last item
first = fruits.pop(0)                # remove and return by index
del fruits[0]                        # delete by index, returns nothing
print(fruits, "| popped:", first, last)

## Searching

| C# | Python |
|---|---|
| `list.Contains(x)` | `x in lst` |
| `list.IndexOf(x)` | `lst.index(x)` (raises `ValueError` if missing, does **not** return -1) |
| `list.Count(i => i == x)` (LINQ) | `lst.count(x)` |

In [ ]:
colors = ["red", "green", "blue", "green"]

print("green" in colors)        # True
print("pink" not in colors)     # True
print(colors.index("green"))    # 1 — first occurrence
print(colors.count("green"))    # 2

## Sorting and reversing

Python has two flavours of sorting:

- `lst.sort()` sorts **in place** and returns `None` (like C#'s `List<T>.Sort()`)
- `sorted(iterable)` returns a **new** list (like LINQ's `OrderBy(...).ToList()`)

The `key=` argument plays the role of the LINQ key selector lambda.

```csharp
// C#
var byLength = words.OrderBy(w => w.Length).ToList();
var desc     = words.OrderByDescending(w => w).ToList();
```

In [ ]:
words = ["banana", "Kiwi", "apple", "fig"]

by_length = sorted(words, key=len)                 # OrderBy(w => w.Length)
desc = sorted(words, reverse=True)                 # OrderByDescending(w => w)
case_insensitive = sorted(words, key=str.lower)    # ordinal-ignore-case-ish
print(by_length)
print(desc)
print(case_insensitive)

words.sort()        # in place
print(words)
words.reverse()     # in place
print(words)

⚠️ **Gotcha:** because `sort()` returns `None`, this is a classic bug:

In [ ]:
nums = [3, 1, 2]
result = nums.sort()
print(result)   # None! Use sorted(nums) if you want a new list.
print(nums)     # the original was sorted in place

## Combining and repeating

In [ ]:
a = [1, 2]
b = [3, 4]

print(a + b)      # concatenation creates a new list (C#: a.Concat(b).ToList())
print(a * 3)      # repetition: [1, 2, 1, 2, 1, 2]
print([0] * 5)    # handy for initialising: like new int[5]

## Iterating

`for` in Python is always a *foreach*. When you also need the index, use `enumerate` instead of a manual counter.

```csharp
// C#
foreach (var (item, i) in items.Select((x, i) => (x, i)))
    Console.WriteLine($"{i}: {item}");
```

In [ ]:
for i, item in enumerate(["x", "y", "z"]):
    print(f"{i}: {item}")

## Gotcha: variables are references (aliasing)

A `List<T>` in C# is a reference type, and a Python list behaves the same way: assignment copies the **reference**, not the list.

In [ ]:
original = [1, 2, 3]
alias = original          # same object!
alias.append(4)
print(original)           # [1, 2, 3, 4]
print(alias is original)  # True — same identity (C#: ReferenceEquals)

To copy, use `.copy()`, `list(x)`, or a full slice `x[:]`. These are **shallow** copies — nested lists are still shared. Use `copy.deepcopy` for a deep copy.

In [ ]:
import copy

grid = [[1, 2], [3, 4]]
shallow = grid.copy()
deep = copy.deepcopy(grid)

grid[0].append(99)
print("shallow:", shallow)   # inner list is shared -> sees 99
print("deep:   ", deep)      # fully independent

⚠️ **Gotcha:** `[[0] * 3] * 3` creates three references to the *same* inner list. Use a comprehension instead (covered in the comprehensions notebook).

In [ ]:
bad = [[0] * 3] * 3
bad[0][0] = 1
print(bad)    # every row changed!

good = [[0] * 3 for _ in range(3)]
good[0][0] = 1
print(good)

## Equality

`==` compares lists **by value** (element by element) — unlike C#, where `==` on two `List<T>` instances compares references and you would need `SequenceEqual`.

In [ ]:
print([1, 2, 3] == [1, 2, 3])   # True  (C#: SequenceEqual)
print([1, 2, 3] is [1, 2, 3])   # False (two different objects)
print([1, 2] < [1, 3])          # lists compare lexicographically

## Type hints

You can annotate list element types. This helps your IDE and type checkers (mypy, pyright), but Python does **not** enforce it at runtime.

In [ ]:
scores: list[int] = [90, 85]
scores.append("not an int")   # runs fine! Only a type checker would complain.
print(scores)

## Summary

| Task | C# | Python |
|---|---|---|
| Create | `new List<int> { 1, 2 }` | `[1, 2]` |
| Length | `.Count` | `len(lst)` |
| Last item | `lst[^1]` | `lst[-1]` |
| Add | `.Add(x)` | `.append(x)` |
| Add many | `.AddRange(xs)` | `.extend(xs)` |
| Contains | `.Contains(x)` | `x in lst` |
| Sort in place | `.Sort()` | `.sort()` |
| Sorted copy | `.OrderBy(k).ToList()` | `sorted(lst, key=k)` |
| Value equality | `.SequenceEqual(other)` | `lst == other` |
| Copy | `new List<T>(lst)` | `lst.copy()` |

## Exercises

1. Create a list of the numbers 1–10, then remove all even numbers **in place**.
2. Given `names = ["bob", "Alice", "carol", "Dave"]`, produce a new list sorted case-insensitively, longest name first.
3. Explain why `matrix = [[0] * 2] * 2` is dangerous.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1 — iterate over a copy, or rebuild in place with a slice assignment
nums = list(range(1, 11))
nums[:] = [n for n in nums if n % 2 != 0]

# 2 — key returns a tuple: (-length, lowercase name)
names = ["bob", "Alice", "carol", "Dave"]
result = sorted(names, key=lambda n: (-len(n), n.lower()))

# 3 — the outer list holds two references to the SAME inner list,
#     so mutating one row mutates "both".
```

</details>